In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

spark = SparkSession.builder \
    .appName("OnlineRetailETL") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-namenode:9000") \
    .getOrCreate()

print("Spark Session created successfully")

df = spark.read.csv(
    "hdfs://hadoop-namenode:9000/user/airflow/data/batches/",
    header=True,
    inferSchema=True
)

print(f"Total Rows: {df.count():,}")
df.printSchema()
df.show(5)

df_clean = df.filter(col("CustomerID").isNotNull()) \
             .filter(col("Quantity") > 0) \
             .filter(col("UnitPrice") > 0)

print(f"Clean Rows: {df_clean.count():,}")

df_clean = df_clean.withColumn(
    "InvoiceDate", 
    to_date(col("InvoiceDate"), "M/d/yyyy H:mm")
)

df_clean.show(5)

spark.stop()
print("Spark Session stopped")

Spark Session created successfully
Total Rows: 541,909
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+
|   577504|    22409|MONEY BOX BISCUIT...|       1|11/20/2011 12:36|     1.25|   14159.0|United Kingdom|
|   577504|    22408|MONEY BOX CONFECT...|       1|11/20/2011 12:36|     1.25|   14159.0|United Kingdom|
|   577504|    22406|MONEY BOX KINGS C...|       1|11/20/2011 12:36|     1.25|   14